# Phase 2: Fine-tune XLM-R (intent + entities)

**Before running:** Runtime -> Change runtime type -> **T4 GPU**.

Run the cells **in order, one at a time**. If you already uploaded and extracted `clinic-assistant.zip` (the folder shows in the left sidebar), skip the upload cell.

In [ ]:
!nvidia-smi | head -n 12

### 1. Upload + extract (skip if the folder is already in the sidebar)

In [ ]:
from google.colab import files
up = files.upload()   # choose clinic-assistant.zip
!unzip -q -o clinic-assistant*.zip

### 2. Find the project folder and move into it
This finds `scripts/generate_data.py` wherever the zip was extracted, so the folder name doesn't matter.

In [ ]:
import os, glob
hits = glob.glob('/content/**/scripts/generate_data.py', recursive=True)
assert hits, 'Could not find scripts/generate_data.py. Check the extracted folder in the left sidebar.'
root = os.path.dirname(os.path.dirname(hits[0]))
%cd $root
!ls
import torch, transformers
print('torch', torch.__version__, '| transformers', transformers.__version__, '| GPU:', torch.cuda.is_available())

### 3. Data (no pip installs needed; Colab already has everything)

In [ ]:
!python scripts/generate_data.py | tail -n 12
!python scripts/validate_handwritten.py

## Classical baseline (CPU, seconds)
TF-IDF + logistic regression, intent only. This is the number the fine-tuned model must beat.

In [ ]:
!python scripts/baseline_tfidf.py

## Train (about 10-20 min on a T4)
Saves the best epoch (by validation score) to `models/nlu`.

In [ ]:
!python scripts/train_nlu.py --out models/nlu --epochs 5

## Evaluate
Prints tables for the synthetic test set and your hand-written set, writes `results/metrics.json` and error files.

In [ ]:
!python scripts/eval_nlu.py --model-dir models/nlu

## Save outputs
Download `results.zip` and send me the contents of `results/metrics.json`. Also copy the model to Drive so you don't retrain.

In [ ]:
!zip -q -r results.zip results models/nlu/meta.json
files.download('results.zip')

In [ ]:
# Optional: keep the trained model (about 1.1 GB) in your Google Drive
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/clinic-assistant && cp -r models/nlu /content/drive/MyDrive/clinic-assistant/